# Fake News Detection — GRU inference

Run this after training `gru.ipynb` (it needs `results/gru_best.h5` and `results/gru_tokenizer.pkl`). Edit the `headline` variable in the last cell and re-run it to test different headlines.

In [1]:
import os
import re
import json
import pickle

import nltk
from nltk.corpus import stopwords
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.sequence import pad_sequences

try:
    STOPWORDS = set(stopwords.words("english"))
except LookupError:
    nltk.download("stopwords")
    STOPWORDS = set(stopwords.words("english"))

RESULTS_DIR = os.path.join(os.getcwd(), "results")


def clean_text(text):
    text = re.sub(r"[^a-zA-Z0-9\s]", " ", str(text).lower()).strip()
    tokens = [t for t in text.split() if t not in STOPWORDS]
    return " ".join(tokens)


In [2]:
model_path = os.path.join(RESULTS_DIR, "gru_best.h5")
results_path = os.path.join(RESULTS_DIR, "gru.json")
tokenizer_path = os.path.join(RESULTS_DIR, "gru_tokenizer.pkl")

if not (os.path.exists(model_path) and os.path.exists(tokenizer_path)):
    raise SystemExit("No trained GRU model found -- run gru.ipynb first.")

model = load_model(model_path)
with open(results_path) as f:
    params = json.load(f)["params"]
with open(tokenizer_path, "rb") as f:
    tokenizer = pickle.load(f)

print("Loaded GRU model. max_len =", params["max_len"])


Loaded GRU model. max_len = 40


In [3]:
def predict(text):
    cleaned = clean_text(text)
    seq = tokenizer.texts_to_sequences([cleaned])
    padded = pad_sequences(seq, maxlen=params["max_len"], padding="post", truncating="post")
    prob = float(model.predict(padded, verbose=0)[0][0])
    label = "FAKE" if prob >= 0.5 else "REAL"
    return label, prob


## Try a headline
Edit the string below and re-run this cell.

In [4]:
headline = "Senate passes bipartisan infrastructure bill after months of negotiation"

label, prob = predict(headline)
print(f"{label}  (fake probability: {prob:.3f})  <-  {headline}")


REAL  (fake probability: 0.011)  <-  Senate passes bipartisan infrastructure bill after months of negotiation
